In [1]:
import os

In [2]:
%pwd

'd:\\Machine_Learning_Projects\\Text-Summerizer\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Machine_Learning_Projects\\Text-Summerizer'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

In [6]:
from TextSummerizer.constants import *
from TextSummerizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH, params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name=config.tokenizer_name
        )

        return data_transformation_config

In [8]:
import os
import ssl
import certifi

cafile = certifi.where()
os.environ["SSL_CERT_FILE"] = cafile
os.environ["REQUESTS_CA_BUNDLE"] = cafile
os.environ["CURL_CA_BUNDLE"] = cafile

# Ensure default SSL context uses certifi bundle (workaround for Windows OpenSSL store issues)
def _create_default_context(purpose=ssl.Purpose.SERVER_AUTH, cafile=None, capath=None, cadata=None):
    ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
    ctx.load_verify_locations(cafile=cafile or certifi.where())
    return ctx

try:
    ssl.create_default_context = _create_default_context
except Exception:
    pass

# Pre-load certs into a context to avoid load_default_certs() failures
try:
    ctx = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
    ctx.load_verify_locations(cafile=cafile)
except Exception as e:
    print('Warning loading certs into SSLContext:', e)


In [11]:
import os
from TextSummerizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

In [15]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):
        input_encodings = self.tokenizer(example_batch['dialogue'], max_length=1024, truncation=True, padding='max_length')

        target_encodings = self.tokenizer(example_batch['summary'], max_length=128, truncation=True, padding='max_length')

        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings.get('attention_mask'),
            'labels': target_encodings['input_ids']
        }

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched=True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir, "samsum_dataset"))


In [16]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-07-23 00:25:54,436]: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-23 00:25:54,462]: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-23 00:25:54,468]: INFO: common: created directory at: artifacts]


[2026-07-23 00:25:54,471]: INFO: common: created directory at: artifacts/data_transformation]


Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 38543.83 examples/s]
